#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Text Classification - Spam Detection (Inglés)

# Carga & Exploración de Datos

In [1]:
#Importar Librerías
import pandas as pd
import numpy as np
import spacy

In [2]:
#Carga de Datos
df = pd.read_csv("/content/spam_classification.csv")

In [3]:
#Verificar Columns
df.columns

Index(['category', 'Message'], dtype='object')

In [4]:
#Verificar Tamaño
df.shape

(5572, 2)

In [5]:
#Verificar Contenido
df.head()

,category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
#Generar Label Spam (Binario)
df['label'] = np.where(df['category'] == 'spam', 1, 0)

In [8]:
#Verificar Distribución Label
df['label'].value_counts(normalize=True)

,proportion
label,
0,0.865937
1,0.134063


# Pre-Procesamiento

In [9]:
#Descargar spaCy pipeline
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
#Cargar Pipeline
nlp = spacy.load("en_core_web_sm")

In [11]:
#Función para Pre-Processing + Tokenización
def get_tokens(text):
  doc = nlp(text)
  tokens = [token.lemma_.lower() for token in doc if token if not token.is_stop and not token.is_punct and not token.like_url and not token.like_email]
  return tokens

# Modelo Bag of Words & LSA

In [12]:
#Importar Librerías
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

In [13]:
#Crear Objeto TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(lowercase=False,
                                   preprocessor=None,
                                   tokenizer=get_tokens)

In [14]:
#Ajustar TfidfVectorizer a Mensajes
tfidf_vectorizer.fit(df['Message'])
print("Tamaño Vocabulario:", len(tfidf_vectorizer.get_feature_names_out()))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Tamaño Vocabulario: 7957


In [15]:
#Generar Representación Vectorial
tfidf_vectors = tfidf_vectorizer.transform(df['Message'])
print("Tamaño Rep. Vectorial:", tfidf_vectors.shape)

Tamaño Rep. Vectorial: (5572, 7957)


In [16]:
#Parametrización & Ajuste Objeto SVD
n_components = 1000
SVD = TruncatedSVD(n_components=n_components,
                   random_state=0)
SVD.fit(tfidf_vectors)

TruncatedSVD(n_components=1000, random_state=0)

In [17]:
#Resumen Varianza Explicada
print("Total Varianza Explicada:", round(SVD.explained_variance_ratio_.sum(),3))

Total Varianza Explicada: 0.736


In [18]:
#Transformación Matriz Rep. Vectorial
tfidf_lsa_vectors = SVD.transform(tfidf_vectors)
print("Tamaño Rep. Vectorial:", tfidf_lsa_vectors.shape)

Tamaño Rep. Vectorial: (5572, 1000)


# Train & Test Sets

In [19]:
#Importar Librerías
from sklearn.model_selection import train_test_split

In [20]:
#Creación Vector Labels
labels = np.array(df['label'])

In [21]:
#Creación Train & Test Set
train_size = 0.8
X_train, X_test, y_train, y_test = train_test_split(tfidf_lsa_vectors, labels,
                                                    train_size=train_size,
                                                    stratify= labels,
                                                    random_state=0)

In [22]:
#Validar Tamaño Datasets
print("Train Set: X", X_train.shape, "- y:", y_train.shape)
print("Test Set: X", X_test.shape, "- y:", y_test.shape)

Train Set: X (4457, 1000) - y: (4457,)
Test Set: X (1115, 1000) - y: (1115,)


# KNN

In [23]:
#Importar Librerías
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

In [25]:
#k-NN (Hyper-Parameter Tuning)
clf_kNN_grid = KNeighborsClassifier()
param_grid_kNN = [{'n_neighbors': list(range(2,10,1))}]

clf_kNN_ht = GridSearchCV(estimator=clf_kNN_grid,
                          param_grid=param_grid_kNN,
                          scoring='f1',
                          n_jobs=-1)

clf_kNN_ht.fit(X_train, y_train)

print("Mejor Hiper-Parámetro:", clf_kNN_ht.best_params_)

Mejor Hiper-Parámetro: {'n_neighbors': 3}


In [26]:
#Entrenamiento K-NN (Mejor Hiper-Parámetro)
clf_kNN = KNeighborsClassifier(n_neighbors=clf_kNN_ht.best_params_['n_neighbors'])
clf_kNN.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

In [27]:
#Validación Cruzada (Train Set)
cross_val_scores = cross_validate(clf_kNN, X_train, y_train, cv=5, scoring=['f1'])
print("Promedio F1 Score (Validation):", round(cross_val_scores['test_f1'].mean(),3))
print("Desviación F1 Score (Validation):", round(cross_val_scores['test_f1'].std(),3))

Promedio F1 Score (Validation): 0.69
Desviación F1 Score (Validation): 0.016


In [28]:
#Predicción Test Set
y_pred_kNN = clf_kNN.predict(X_test)

In [29]:
#Evaluación Test Set
print("Matriz Confusión:\n", confusion_matrix(y_test, y_pred_kNN))
print("F1 Score (Test):", round(f1_score(y_test, y_pred_kNN),3))

Matriz Confusión:
 [[964   2]
 [ 76  73]]
F1 Score (Test): 0.652
